# Task 2.7 — Tuning MLP trên holdout seed 9

**Inputs:** competition data + output task **2.6**. Bật Kaggle GPU.
Chỉ dùng train/validation; không đọc test hoặc leaderboard. Tối đa 16 cấu hình mới; cấu hình baseline 2.6 luôn có mặt trong bảng.
Tuning tuần tự để giảm số lượt: learning rate/batch → kiến trúc/dropout → regularization/weighting.
AUC holdout dùng sàng lọc; chưa được coi là bằng chứng cải thiện trên dữ liệu mới.


## 2.7.1 — Nạp engine, kiểm tra đúng dataset và membership

Nạp code đã hash từ 2.6, kiểm tra train checksum và giữ nguyên split 80/20 seed 9.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import hashlib
import json

DATA_DIR = None  # Auto-detect inputs; set a directory here only to disambiguate.

def resolve_data_dir(data_dir=None, expected_sha256=None,
                     required_files=("train.csv",), input_root="/kaggle/input"):
    """Find attached data without assuming Kaggle's mount layout."""
    from pathlib import Path
    import hashlib

    root = Path(input_root)
    if data_dir is not None:
        candidates = [Path(data_dir)]
        missing = [name for name in required_files if not (candidates[0] / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"DATA_DIR={data_dir} is missing {missing}. "
                "Attach competition data or set DATA_DIR=None for automatic discovery."
            )
    else:
        candidates = sorted({
            path.parent for path in root.rglob("train.csv")
            if path.is_file() and all((path.parent / name).is_file() for name in required_files)
        })
        if not candidates:
            mounted = sorted(path.name for path in root.iterdir()) if root.is_dir() else []
            raise FileNotFoundError(
                f"No attached directory contains {list(required_files)} under {root}. "
                "Use Add Input to attach competition playground-series-s6e11. "
                f"Mounted entries: {mounted}"
            )

    if expected_sha256 is not None:
        matching = []
        for directory in candidates:
            digest = hashlib.sha256()
            with (directory / "train.csv").open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() == expected_sha256:
                matching.append(directory)
        if not matching:
            raise ValueError(
                "Attached train.csv does not match the task 2.6 checksum. "
                f"Check the competition data/version. Candidates: {candidates}"
            )
        candidates = matching
    elif len(candidates) > 1:
        bundles = [directory for directory in candidates
                   if all((directory / name).is_file()
                          for name in ("train.csv", "test.csv", "sample_submission.csv"))]
        if bundles:
            candidates = bundles

    if len(candidates) != 1:
        raise ValueError(
            f"Multiple valid data directories found: {candidates}. "
            "Set DATA_DIR to the exact directory you want to use."
        )
    return candidates[0]

# Nếu gắn nhiều version 2.6, điền chính xác thư mục version cần dùng.
FOUNDATION_DIR = None
if FOUNDATION_DIR is None:
    found = sorted(Path("/kaggle/input").rglob("foundation_manifest.json"))
    if len(found) != 1:
        raise FileNotFoundError(f"Attach exactly one task 2.6 output; found: {found}")
    FOUNDATION_DIR = found[0].parent
else:
    FOUNDATION_DIR = Path(FOUNDATION_DIR)
manifest = json.loads((FOUNDATION_DIR / "foundation_manifest.json").read_text(encoding="utf-8"))
engine_file = FOUNDATION_DIR / "credit_fraud_dl_engine.py"
if hashlib.sha256(engine_file.read_bytes()).hexdigest() != manifest["engine_sha256"]:
    raise ValueError("Foundation engine checksum mismatch.")
spec = importlib.util.spec_from_file_location("credit_fraud_dl_engine", engine_file)
dl = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dl
spec.loader.exec_module(dl)

import numpy as np
import pandas as pd
DATA_DIR = resolve_data_dir(DATA_DIR, expected_sha256=manifest["train_sha256"])
print("Resolved DATA_DIR:", DATA_DIR)
train_path = DATA_DIR / "train.csv"
dl.verify_source(train_path, manifest)
train = pd.read_csv(train_path)
dl.validate_train(train)
for name, digest in manifest["membership_sha256"].items():
    if dl.file_hash(FOUNDATION_DIR / name) != digest:
        raise ValueError(f"Foundation membership file changed: {name}")
split = pd.read_csv(FOUNDATION_DIR / "holdout_split_seed9.csv")
train_idx, valid_idx = dl.holdout_indices(train, split)
print("Foundation:", manifest["foundation_id"])
print("Runtime:", dl.environment())


## 2.7.2 — Hàm ghi nhận thí nghiệm

Config ID phụ thuộc toàn bộ cấu hình. Mỗi cấu hình có thư mục riêng; không ghi đè baseline lịch sử. Run đã hoàn tất chỉ reuse khi signature và checksum hợp lệ. Lưu bảng sau mỗi lượt để còn kết quả nếu phiên Kaggle bị ngắt.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/task_2_7_mlp_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_NEW_CONFIGS = 16
reference = dl.json_read(FOUNDATION_DIR / "baseline_reference.json")
if reference["foundation_id"] != manifest["foundation_id"]:
    raise ValueError("Baseline belongs to a different foundation.")
baseline = reference["rerun"]
results = {baseline["config_id"]: baseline}
new_ids = set()

def save_tuning():
    ordered = dl.ranked(list(results.values()))
    table = pd.DataFrame([{
        "config_id": r["config_id"], "family": r["config"]["family"],
        "roc_auc": r["roc_auc"], "average_precision": r["average_precision"],
        "best_epoch": r["best_epoch"], "training_seconds": r["training_seconds"],
        "delta_vs_baseline": r["roc_auc"] - baseline["roc_auc"],
    } for r in ordered])
    table.to_csv(OUTPUT_DIR / "experiments.csv", index=False)
    dl.json_write(OUTPUT_DIR / "tuning_manifest.json", {
        "foundation_id": manifest["foundation_id"], "source_sha256": manifest["train_sha256"],
        "selection": "Shared seed-9 holdout ROC-AUC; AP as tie-break; requires CV confirmation.",
        "results": ordered, "best_config": ordered[0]["config"],
    })
    return table

def run_mlp(cfg):
    key = dl.config_id(cfg)
    if key not in results:
        if len(new_ids) >= MAX_NEW_CONFIGS:
            raise RuntimeError("Configured experiment budget exceeded.")
        new_ids.add(key)
        results[key] = dl.train_run(
            train, train_idx, valid_idx, cfg, OUTPUT_DIR / key, manifest["train_sha256"])
        save_tuning()
    return results[key]


## 2.7.3 — Learning rate và batch size

Giữ hidden layers 128/64. Tăng trần epoch lên 50 nhưng dừng sớm patience 8; scheduler giảm LR khi AUC chững. Không có nghĩa mọi lượt đều train đủ 50 epoch. Các thay đổi kết hợp trong bước sàng lọc này sẽ được kiểm chứng bằng baseline scheduler-off ở bước sau.


In [ ]:
start_config = dl.config(max_epochs=50, patience=8, scheduler=True)
stage1 = []
for learning_rate, batch_size in [
    (0.0003, 2048), (0.001, 2048), (0.003, 2048),
    (0.001, 1024), (0.001, 4096),
]:
    cfg = dl.config(**{**start_config, "learning_rate": learning_rate, "batch_size": batch_size})
    stage1.append(run_mlp(cfg))
best_stage1 = dl.ranked(stage1)[0]["config"]
display(save_tuning())


## 2.7.4 — Kích thước mạng và dropout

Thử 4 cấu hình dễ giải thích: mạng gốc ít dropout, mạng rộng hơn, mạng sâu hơn và không dropout. Đổi từng nhóm để bảng thí nghiệm thể hiện được nguyên nhân.


In [ ]:
stage2 = []
for hidden, dropout in [
    ([128, 64], [0.10, 0.05]),
    ([256, 128], [0.10, 0.05]),
    ([256, 128, 64], [0.10, 0.10, 0.05]),
    ([128, 64], [0.0, 0.0]),
]:
    cfg = dl.config(**{**best_stage1, "hidden": hidden, "dropout": dropout})
    stage2.append(run_mlp(cfg))
best_architecture = dl.ranked(stage1 + stage2)[0]["config"]
display(save_tuning())


## 2.7.5 — Weight decay, scheduler và class weighting

So sánh 3 mức weight decay, thử tắt scheduler nhưng giữ cùng giới hạn epoch, và thêm class weight vừa phải hoặc balanced. Balanced tính tỷ lệ từ đúng train split, không nhìn validation. Chọn baseline nếu các cấu hình mới không thắng.


In [ ]:
for decay in [0.0, 0.00001, 0.0001]:
    run_mlp(dl.config(**{**best_architecture, "weight_decay": decay}))
run_mlp(dl.config(**{**best_architecture, "scheduler": False}))
for weight in [3.0, "balanced"]:
    run_mlp(dl.config(**{**best_architecture, "positive_weight": weight}))
display(save_tuning())
print("New configurations:", len(new_ids), "/", MAX_NEW_CONFIGS)
print("Shortlist:", [r["config_id"] for r in dl.ranked(list(results.values()))[:3]])
